# Purdue University Affiliation Analysis

This notebook analyzes awardee names for Purdue University connections using the Institution Checker pipeline.

**Workflow:**
1. **Setup**: Configure environment, reload modules, and set API keys.
2. **Data Loading**: Load and prepare the dataset (Full or Sample).
3. **Clustering**: Group similar names to optimize processing.
4. **Validation**: Run a quick random smoke test for sanity.
5. **Execution**: Run the full pipeline on the clustered dataset.
6. **Analysis**: Review results, optionally run unbiased recovery on expected-missed names, and export data.

In [1]:
# 1. SETUP & CONFIGURATION
# -----------------------------------------------------------------------------
# This cell handles module reloading and API configuration.
# Run this FIRST to ensure the environment is ready.

import sys
import importlib
import pandas as pd
import time
from pathlib import Path

# --- Fresh-run state reset (avoid stale notebook variables) ---
CURRENT_RUN_ID = None
RUN_RESULTS_READY = False
for var_name in [
    "df", "df_base", "df_vips", "df_others", "df_random", "df_clustered",
    "cluster_reps", "cluster_results", "cluster_institution_map", "final_results"
]:
    if var_name in globals():
        del globals()[var_name]

# --- Force Local Source Path (prevents stale UNC/site-package imports) ---
workspace_src = Path(r"e:\Awards DB Code\step3_institution_checker\src")
if workspace_src.exists():
    sys.path = [p for p in sys.path if "institution_checker" not in p.lower()]
    sys.path.insert(0, str(workspace_src))
    print(f"Using local source path: {workspace_src}")

# Drop cached institution_checker modules so imports resolve from the path above.
for mod in [m for m in list(sys.modules) if m == "institution_checker" or m.startswith("institution_checker.")]:
    del sys.modules[mod]

# --- Module Reloading (Development Mode) ---
# Reinstall local package to ensure latest changes are applied
!pip uninstall -y institution_checker
!pip install -e .

# Force reload of modules to pick up changes
modules_to_reload = [
    'institution_checker.config',
    'institution_checker.search',
    'institution_checker.llm_processor',
    'institution_checker.main',
    'institution_checker'
]

for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"Reloaded {module_name}")

# --- Imports (AFTER Reloading) ---
# Import AFTER reloading to ensure we get the new function objects
import institution_checker
import institution_checker.llm_processor
import institution_checker.search
from institution_checker import (
    run_pipeline,
    resolve_names,
    set_api_key,
    expand_cluster_results_identity_safe,
    run_targeted_recovery_pass,
 )
from institution_checker.llm_processor import enable_prompt_logging
from name_matcher.pipeline import NameMatcher, NameMatcherConfig

print(f"institution_checker loaded from: {institution_checker.__file__}")

# --- API Configuration ---
API_KEY = "sk-3f1dbbf2450e46ab9541dffba4f18ec6"
set_api_key(API_KEY)

# Configure NameMatcher
config = NameMatcherConfig(name_column="name", use_tqdm=True)
config.llm_token = API_KEY

print("\nSetup complete:")
print("   - Fresh in-memory state initialized")
print("   - Modules reloaded and installed")
print("   - API keys configured")
print("   - Browser timeout fixes active (15s)")
print("   - Search strategies optimized (DuckDuckGo prioritized + Rate Limit Protection)")
print("   - Evidence gathering expanded (8 excerpts)")

Using local source path: e:\Awards DB Code\step3_institution_checker\src
Found existing installation: institution-checker 0.1.0
Uninstalling institution-checker-0.1.0:
  Successfully uninstalled institution-checker-0.1.0
Obtaining file:///E:/Awards%20DB%20Code/step3_institution_checker
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for institution-checker (pyproject.toml): started
  Building editable for institution-checker (pyproject.toml): finished with status 'done'
  Created wheel for institution-checker: filename=institu

In [2]:
# 2. DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
# Select the dataset to process and prepare it for analysis.

# --- Configuration ---
DATASET_OPTION = "purdue"  # Options: "purdue", "nobel", "sample", "accuracy_test_25"

# File Paths
PATHS = {
    "purdue": r"E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx",
    "nobel": r"E:\Ace\Nobel Only Test Set.xlsx"
}

# --- Load Data ---
if DATASET_OPTION == "sample":
    print("ðŸ“‰ CREATING TEST SAMPLE: 9 VIPs + 100 Random Names")
    # Load base dataset (Nobel) for sampling
    df_base = pd.read_excel(PATHS["nobel"], dtype=str)
    
    vip_names = [
        "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
        "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
        "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
    ]
    
    vip_mask = df_base['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
    df_vips = df_base[vip_mask].copy()
    df_others = df_base[~vip_mask].copy()
    
    sample_size = min(100, len(df_others))
    df_random = df_others.sample(n=sample_size, random_state=42)
    
    df = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)
    print(f"âœ… Sample created: {len(df)} records")

elif DATASET_OPTION == "accuracy_test_25":
    print("ðŸ“‰ CREATING ACCURACY TEST: 9 VIPs + 16 Random Non-Connected")
    df_base = pd.read_excel(PATHS["nobel"], dtype=str)
    
    vip_names = [
        "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
        "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
        "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
    ]
    
    # 1. Get VIPs
    vip_mask = df_base['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
    df_vips = df_base[vip_mask].drop_duplicates(subset=['name']).copy()
    
    # 2. Get Others (Non-VIPs)
    df_others = df_base[~vip_mask].copy()
    
    # 3. Select 16 random names
    # Using a fixed seed ensures reproducibility. 
    df_random = df_others.sample(n=16, random_state=123) 
    
    # Combine
    df = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)
    
    print(f"âœ… Test set created: {len(df)} records")
    print(f"   VIPs (Positive Controls): {len(df_vips)}")
    print(f"   Others (Negative Controls): {len(df_random)}")
    
    print("\nðŸ“‹ List of names to check:")
    print("-" * 30)
    for i, name in enumerate(df['name'], 1):
        role = "VIP" if any(v in name for v in vip_names) else "Other"
        print(f"{i:2}. {name:<30} [{role}]")

elif DATASET_OPTION in PATHS:
    path = PATHS[DATASET_OPTION]
    df = pd.read_excel(path, dtype=str)
    print(f"âœ… Loaded {len(df):,} records from {DATASET_OPTION.upper()} dataset")
    print(f"   Source: {path}")

else:
    raise ValueError(f"Invalid DATASET_OPTION: {DATASET_OPTION}")

# --- Clustering ---
print("\nðŸ§© Clustering similar names...")
matcher = NameMatcher(config)
result = matcher.cluster(df)
df_clustered = result.dataframe
print(f"âœ… Clustering complete: {len(df)} names -> {df_clustered['cluster_label'].nunique()} clusters")


âœ… Loaded 607 records from PURDUE dataset
   Source: E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx

ðŸ§© Clustering similar names...
--- Name Matcher Process Started (v6.8.8 Improved Canonical Name Selection) ---

1. Loading and preparing data...
   Loaded 607 names. Done in 0.02s
2. Vectorizing names for similarity scoring...
   Done in 0.02s
3. Generating candidate pairs via Blocking...
   Generated 682 candidate pairs from blocking.
   Done in 0.00s
4. Applying Stricter Cluster-Centric Filtering...


   Filtering Pairs: 100%|██████████| 682/682 [00:00<?, ?pair/s]


   Filter Stats: {'merged_expansion': 299, 'rejected_transitive_conflict': 45}
   Done in 0.00s
5. Reviewing ambiguous pairs with LLM...
   Sending 6 unique pairs to LLM for review using 5 parallel workers...


   LLM Review: 100%|██████████| 1/1 [00:04<00:00,  4.30s/batch]

   Done in 4.30s
6. Finalizing cluster data structures...
   Done in 0.00s
7. Refining clusters by ejecting outliers...
   Found and ejected 0 outliers from clusters.
   Done in 0.10s
8. Assigning canonical names and saving results...

--- Results Summary ---
   - Total names processed: 607
   - Unique entities (clusters) found: 305

   --- Sample of Largest Clusters Found ---
   Cluster 1 (Size: 10): 'Seymour Benzer'
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - ...
   Cluster 2 (Size: 8): 'Alvin Carl Plantinga'
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - ...
   Cluster 3 (Size: 7): 'George Andrew Olah'
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - ...
   Cluster 4 (Size: 7): 'Hans Albrecht Bethe'
     - H. A. Bethe
     - H. A. Bethe
     - Hans A. Beth

In [3]:
# 3. PIPELINE VALIDATION (RANDOM SMOKE TEST)
# -----------------------------------------------------------------------------
# Run a quick random sanity check before full execution.

SMOKE_TEST_SIZE = 9

available_names = df['name'].dropna().astype(str).str.strip()
available_names = available_names[available_names != '']

if available_names.empty:
    raise RuntimeError('No names available for smoke test.')

sample_n = min(SMOKE_TEST_SIZE, len(available_names))
smoke_test_names = available_names.sample(n=sample_n, random_state=42).tolist()

print(f"Random smoke test ({sample_n} names)")
print("=" * 60)
print("Testing pipeline behavior on unbiased random names...")

start_time = time.time()

smoke_results = await run_pipeline(
    smoke_test_names,
    batch_size=3,
    use_enhanced_search=True,
    dataset_profile="high_connection",
    use_dynamic_batching=False
)

elapsed = time.time() - start_time

print(f"\nSMOKE TEST RESULTS ({elapsed:.1f}s, {elapsed/sample_n:.1f}s per name):")
print("-" * 60)

connected_count = 0
for name, res in zip(smoke_test_names, smoke_results):
    verdict = str(res.get('verdict', 'unknown')).lower()
    institution = str(res.get('institution', ''))
    is_purdue = 'purdue' in institution.lower()
    is_connected = verdict == 'connected' and is_purdue

    if is_connected:
        connected_count += 1
        icon = "✅"
    else:
        icon = "❌"

    print(f"   {icon} {name}: {verdict}")

print(f"\nConnected-to-Purdue in random smoke test: {connected_count}/{sample_n}")
print("Proceed to Section 4 for the full clustered run.")

Random smoke test (9 names)
Testing pipeline behavior on unbiased random names...
[PIPELINE] Starting: 9 name(s) in 3 batch(es) using enhanced search
[PIPELINE] Batch size: 3, Inter-batch delay: 0.5s


Processing batches:   0%|          | 0/3 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/3 =====
[PIPELINE] Names in this batch: ['Vernon W. Ruttan', 'Jay Hopler', 'Chintamani Nagesa Ramachandra Rao']
[BATCH] Processing 3 names: Vernon W. Ruttan, Jay Hopler, Chintamani Nagesa Ramachandra Rao
[BATCH] Phase 1: Running searches in parallel for all 3 names (max 16 concurrent)
[BATCH] Phase 2: Streaming LLM as each search completes (max 6 concurrent)
[BATCH] Search succeeded for Jay Hopler: 20 results (mode=basic_only, queries=1, bucket=plausible, t_basic=2.2s, t_rescue=0.0s, t_enh=0.0s, t_total=2.2s)
[PROGRESS] Starting LLM analysis for: Jay Hopler
[BATCH] Search succeeded for Chintamani Nagesa Ramachandra Rao: 20 results (mode=basic_only, queries=1, bucket=plausible, t_basic=3.4s, t_rescue=0.0s, t_enh=0.0s, t_total=3.4s)
[PROGRESS] Starting LLM analysis for: Chintamani Nagesa Ramachandra Rao
[BATCH] Search succeeded for Vernon W. Ruttan: 28 results (mode=basic_plus_rescue, queries=2, bucket=plausible, t_basic=3.8s, t_rescue=1.6s, t_enh=0.0s, t_total=

In [4]:
# 4. MAIN EXECUTION (FULL DATASET)
# -----------------------------------------------------------------------------
# Run the affiliation checker on the full clustered dataset using the optimized library pipeline.

TEST_MODE = False

# Profile selection by dataset intent:
# - purdue/sample/accuracy_test_25: higher expected positives -> prioritize recall
# - nobel: sparse positives -> optimize throughput
if DATASET_OPTION in {"purdue", "sample", "accuracy_test_25"}:
    DATASET_PROFILE = "high_connection"
    USE_DYNAMIC_BATCHING = False
    USE_ENHANCED_SEARCH = True
    BATCH_SIZE = 12
else:
    DATASET_PROFILE = "low_connection"
    USE_DYNAMIC_BATCHING = True
    USE_ENHANCED_SEARCH = False
    BATCH_SIZE = 30

CURRENT_RUN_ID = int(time.time())
RUN_RESULTS_READY = False

cluster_reps = df_clustered.groupby('cluster_label')['name'].first().reset_index()
cluster_reps.columns = ['cluster_label', 'representative_name']

if TEST_MODE:
    cluster_reps = cluster_reps.head(20)
    print(f"TEST MODE: Checking {len(cluster_reps)} clusters")
else:
    print(f"Checking all {len(cluster_reps)} clusters")

names_to_check = cluster_reps['representative_name'].tolist()

print("Starting Optimized Institution Algorithm (internal library v2)...")
print(f"   Dataset option: {DATASET_OPTION}")
print(f"   Profile: {DATASET_PROFILE}")
print(f"   Search Batch: {BATCH_SIZE}")
print(f"   Dynamic batching: {USE_DYNAMIC_BATCHING}")
print(f"   Enhanced search: {USE_ENHANCED_SEARCH}")
print(f"   Names to process: {len(names_to_check)}")
print(f"   Run ID: {CURRENT_RUN_ID}")
print("-" * 60)

start_time = time.time()
cluster_results = await run_pipeline(
    names_to_check,
    batch_size=BATCH_SIZE,
    use_enhanced_search=USE_ENHANCED_SEARCH,
    dataset_profile=DATASET_PROFILE,
    use_dynamic_batching=USE_DYNAMIC_BATCHING,
)

elapsed = time.time() - start_time
avg_per_name = elapsed / len(names_to_check) if names_to_check else 0
print(f"\nTIMING: total={elapsed:.1f}s | avg={avg_per_name:.2f}s/name")
print(f"THROUGHPUT: {60/avg_per_name:.1f} names/min" if avg_per_name > 0 else "THROUGHPUT: N/A")

fields_to_map = [
    'institution', 'verdict', 'relationship_type', 'relationship_timeframe',
    'verification_detail', 'summary', 'primary_source', 'verification_status',
    'temporal_context', 'confidence', 'purdue_connection',
    'pre_llm_bucket', 'pre_llm_stage', 'pre_llm_score', 'pre_llm_summary',
    'pre_llm_reason_codes', 'pre_llm_used_rescue_query'
]

df_clustered, cluster_institution_map = await expand_cluster_results_identity_safe(
    df_clustered,
    cluster_reps,
    cluster_results,
    batch_size=BATCH_SIZE,
    dataset_profile=DATASET_PROFILE,
    fields_to_map=fields_to_map,
    debug=False,
)

final_results = df_clustered.copy()
final_results['_run_id'] = CURRENT_RUN_ID
final_results['_run_generated_at'] = pd.Timestamp.utcnow().isoformat()
RUN_RESULTS_READY = True

if 'pre_llm_stage' in final_results.columns:
    print("Pre-LLM stage distribution:")
    print(final_results['pre_llm_stage'].fillna('(missing)').value_counts().head(10))

print("Affiliation check complete")
print(f"Fresh rows: {len(final_results)}")

Checking all 305 clusters
Starting Optimized Institution Algorithm (internal library v2)...
   Dataset option: purdue
   Profile: high_connection
   Search Batch: 12
   Dynamic batching: False
   Enhanced search: True
   Names to process: 305
   Run ID: 1775829025
------------------------------------------------------------
[PIPELINE] Starting: 305 name(s) in 26 batch(es) using enhanced search
[PIPELINE] Batch size: 12, Inter-batch delay: 0.5s


Processing batches:   0%|          | 0/26 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/26 =====
[PIPELINE] Names in this batch: ['A. Leon Higginbotham', 'A. Stephen Morse', 'Abraham Tesser', 'Akasha Gloria Hull', 'Akira Suzuki', 'Alan Jay Perlis', 'Albert W. Overhauser', 'Alexandra Boltasseva', 'Alfred E. Mann', 'Alice H. Eagly', 'Alondra Nelson', 'Alvin C. Plantinga']
[BATCH] Processing 12 names: A. Leon Higginbotham, A. Stephen Morse, Abraham Tesser, Akasha Gloria Hull, Akira Suzuki, Alan Jay Perlis, Albert W. Overhauser, Alexandra Boltasseva, Alfred E. Mann, Alice H. Eagly, Alondra Nelson, Alvin C. Plantinga
[BATCH] Phase 1: Running searches in parallel for all 12 names (max 16 concurrent)
[BATCH] Phase 2: Streaming LLM as each search completes (max 6 concurrent)
[BATCH] Search succeeded for Akira Suzuki: 20 results (mode=basic_only, queries=1, bucket=plausible, t_basic=2.9s, t_rescue=0.0s, t_enh=0.0s, t_total=2.9s)
[PROGRESS] Starting LLM analysis for: Akira Suzuki
[BATCH] Search succeeded for Alvin C. Plantinga: 14 results (mode=basic_only,

Retrying failed batches:   0%|          | 0/1 [00:00<?, ?batch/s]

[RETRY] Processing batch 1/1: ['Henri Dorra', 'Herman B Wells', 'Thomas S. Huang']
[BATCH] Processing 3 names: Henri Dorra, Herman B Wells, Thomas S. Huang
[BATCH] Phase 1: Running searches in parallel for all 3 names (max 16 concurrent)
[BATCH] Phase 2: Streaming LLM as each search completes (max 6 concurrent)
[BATCH] Search succeeded for Henri Dorra: 15 results (mode=basic_only, queries=1, bucket=plausible, t_basic=3.2s, t_rescue=0.0s, t_enh=0.0s, t_total=3.2s)
[PROGRESS] Starting LLM analysis for: Henri Dorra
[BATCH] Search succeeded for Herman B Wells: 44 results (mode=basic_plus_enhanced, queries=2, bucket=borderline, t_basic=3.2s, t_rescue=0.0s, t_enh=7.8s, t_total=11.0s)
[PROGRESS] Starting LLM analysis for: Herman B Wells
[BATCH] Search failed for Thomas S. Huang: 
[BATCH] Phase 1 completed in 15.9s
[BATCH] Search telemetry: avg_queries=1.50, avg_attempts=1.50, rescue=0, enhanced=1, slow_fallback=0
[BATCH] Survey counts: hard_no=0, borderline=1, plausible=1, rescue_used=0, resc

Processing batches:   0%|          | 0/1 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/1 =====
[PIPELINE] Names in this batch: ['Arden L. Bement, Jr.', 'Henry L. Roediger III', 'Henry L. Roediger, III', 'Paul Erdos', 'Stephen D. Bechtel', 'Stephen Davison Bechtel']
[BATCH] Processing 6 names: Arden L. Bement, Jr., Henry L. Roediger III, Henry L. Roediger, III, Paul Erdos, Stephen D. Bechtel, Stephen Davison Bechtel
[BATCH] Phase 1: Running searches in parallel for all 6 names (max 16 concurrent)
[BATCH] Phase 2: Streaming LLM as each search completes (max 6 concurrent)
[BATCH] Search succeeded for Henry L. Roediger, III: 17 results (mode=basic_only, queries=1, bucket=plausible, t_basic=3.1s, t_rescue=0.0s, t_enh=0.0s, t_total=3.1s)
[PROGRESS] Starting LLM analysis for: Henry L. Roediger, III
[BATCH] Search succeeded for Henry L. Roediger III: 18 results (mode=basic_only, queries=1, bucket=plausible, t_basic=3.3s, t_rescue=0.0s, t_enh=0.0s, t_total=3.3s)
[PROGRESS] Starting LLM analysis for: Henry L. Roediger III
[BATCH] Search succeeded for Paul

e:\Awards DB Code\step3_institution_checker\src\institution_checker\main.py:3045: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_values = merged_values.fillna(False).astype(bool)


In [5]:
# 5. ANALYSIS, OPTIONAL RECOVERY, & EXPORT (FRESH RUN ONLY)
# -----------------------------------------------------------------------------
# Analyze results from this kernel's latest run only.

# Optional: add expected connected names to recover if they were missed.
EXPECTED_CONNECTED_NAMES = []

EXPORT_FILENAME = 'institution_check_results.xlsx'

if 'final_results' not in globals():
    raise RuntimeError("No in-memory results found. Run Section 4 first.")
if not globals().get('RUN_RESULTS_READY', False):
    raise RuntimeError("Results are not marked fresh. Re-run Section 4 in this kernel.")
if 'CURRENT_RUN_ID' not in globals() or CURRENT_RUN_ID is None:
    raise RuntimeError("Missing CURRENT_RUN_ID. Re-run Section 1 then Section 4.")
if '_run_id' not in final_results.columns:
    raise RuntimeError("Result metadata missing (_run_id). Re-run Section 4.")

run_ids = set(final_results['_run_id'].dropna().astype(int).unique().tolist())
if run_ids != {int(CURRENT_RUN_ID)}:
    raise RuntimeError(
        f"Stale/mixed results detected: expected run_id={CURRENT_RUN_ID}, found={sorted(run_ids)}. Re-run Section 4."
    )

if EXPECTED_CONNECTED_NAMES:
    expected_df = final_results[final_results['name'].isin(EXPECTED_CONNECTED_NAMES)].copy()
    pre_missing = expected_df[expected_df['purdue_connection'] != True]['name'].dropna().astype(str).tolist()
    print(f"Expected connected names provided: {len(EXPECTED_CONNECTED_NAMES)}")
    print(f"Missing before recovery: {len(pre_missing)}")

    if pre_missing:
        recovered = await run_targeted_recovery_pass(pre_missing, max_concurrency=4, debug=False)
        recovered_map = {str(r.get('name', '')).strip(): r for r in recovered if isinstance(r, dict)}

        updatable_fields = [
            'institution', 'verdict', 'relationship_type', 'relationship_timeframe',
            'verification_detail', 'summary', 'primary_source', 'verification_status',
            'temporal_context', 'confidence', 'pre_llm_bucket', 'pre_llm_stage',
            'pre_llm_score', 'pre_llm_summary', 'pre_llm_reason_codes', 'pre_llm_used_rescue_query'
        ]

        applied = 0
        for idx, row in final_results.iterrows():
            row_name = str(row.get('name', '')).strip()
            rec = recovered_map.get(row_name)
            if not rec:
                continue

            verdict = str(rec.get('verdict', '')).lower()
            institution_text = str(rec.get('institution', '')).lower()
            summary_text = str(rec.get('summary', '')).lower()
            detail_text = str(rec.get('verification_detail', '')).lower()
            connected_to_purdue = (
                verdict == 'connected'
                and ('purdue' in institution_text or 'purdue' in summary_text or 'purdue' in detail_text)
            )

            if not connected_to_purdue:
                continue

            for f in updatable_fields:
                if f in final_results.columns and f in rec:
                    final_results.at[idx, f] = rec.get(f)
            if 'purdue_connection' in final_results.columns:
                final_results.at[idx, 'purdue_connection'] = True
            applied += 1

        print(f"Recovery overrides applied: {applied}")

print(f"RESULTS ANALYSIS (run_id={CURRENT_RUN_ID})")
print("=" * 70)

total_records = len(final_results)
purdue_records = final_results[final_results['purdue_connection'] == True]
percentage = (len(purdue_records) / total_records * 100) if total_records > 0 else 0

print(f"   Total records: {total_records:,}")
print(f"   Purdue-affiliated: {len(purdue_records):,} ({percentage:.1f}%)")

RESULTS ANALYSIS (run_id=1775829025)
   Total records: 607
   Purdue-affiliated: 427 (70.3%)


In [6]:
print(f"Run ID: {CURRENT_RUN_ID}")
print(f"Total records: {total_records:,}")
print(f"Purdue-affiliated records: {len(purdue_records):,} ({percentage:.2f}%)")

final_results.to_excel("E:/Ace/Purdue Full Analysis 4.xlsx", index=False)

Run ID: 1775829025
Total records: 607
Purdue-affiliated records: 427 (70.35%)


In [7]:
# 7. GOLDEN RECORD COMPARISON (SOURCE 'Purdue' COLUMN)
# -----------------------------------------------------------------------------
# Compare pipeline output against the original dataset's golden label column: Purdue (True/False).

from pathlib import Path
import numpy as np

if 'final_results' not in globals():
    raise RuntimeError("final_results not found. Run Cell 5 first.")
if 'PATHS' not in globals() or 'purdue' not in PATHS:
    raise RuntimeError("PATHS['purdue'] not available. Run Cell 2 first.")

source_path = Path(PATHS['purdue'])
if not source_path.exists():
    raise FileNotFoundError(f"Source file not found: {source_path}")

golden_df = pd.read_excel(source_path, dtype=str).copy()
if 'name' not in golden_df.columns:
    raise RuntimeError("Golden file is missing required column: 'name'")
if 'Purdue' not in golden_df.columns:
    raise RuntimeError("Golden file is missing required column: 'Purdue'")

def _norm_name(v):
    return ' '.join(str(v).strip().lower().split())

def _to_bool_label(v):
    s = str(v).strip().lower()
    if s in {'true', 't', '1', 'yes', 'y'}:
        return True
    if s in {'false', 'f', '0', 'no', 'n', '', 'nan', 'none', 'null'}:
        return False
    return False

golden_df['name_key'] = golden_df['name'].map(_norm_name)
golden_df['gold_purdue'] = golden_df['Purdue'].map(_to_bool_label)

# Keep one golden label per normalized name (first occurrence).
golden_unique = golden_df.dropna(subset=['name_key']).drop_duplicates(subset=['name_key'], keep='first')[['name_key', 'name', 'gold_purdue', 'Purdue']]

results_df = final_results.copy()
if 'name' not in results_df.columns:
    raise RuntimeError("final_results is missing required column: 'name'")
if 'purdue_connection' not in results_df.columns:
    raise RuntimeError("final_results is missing required column: 'purdue_connection'")

results_df['name_key'] = results_df['name'].map(_norm_name)
results_df['pred_purdue'] = results_df['purdue_connection'].fillna(False).astype(bool)

# Keep one prediction per normalized name (if duplicates exist, positive wins).
pred_unique = (
    results_df.groupby('name_key', as_index=False)['pred_purdue']
    .max()
    .merge(results_df[['name_key', 'name']].drop_duplicates('name_key'), on='name_key', how='left')
)

comparison = golden_unique.merge(
    pred_unique[['name_key', 'pred_purdue']],
    on='name_key',
    how='left'
).copy()
comparison['pred_purdue'] = comparison['pred_purdue'].fillna(False).astype(bool)

tp = int(((comparison['gold_purdue'] == True) & (comparison['pred_purdue'] == True)).sum())
tn = int(((comparison['gold_purdue'] == False) & (comparison['pred_purdue'] == False)).sum())
fp = int(((comparison['gold_purdue'] == False) & (comparison['pred_purdue'] == True)).sum())
fn = int(((comparison['gold_purdue'] == True) & (comparison['pred_purdue'] == False)).sum())

total = len(comparison)
accuracy = ((tp + tn) / total * 100.0) if total else 0.0
precision = (tp / (tp + fp) * 100.0) if (tp + fp) else 0.0
recall = (tp / (tp + fn) * 100.0) if (tp + fn) else 0.0
specificity = (tn / (tn + fp) * 100.0) if (tn + fp) else 0.0
f1 = (2 * tp / (2 * tp + fp + fn) * 100.0) if (2 * tp + fp + fn) else 0.0

print("GOLDEN RECORD COMPARISON (Purdue column)")
print("=" * 70)
print(f"Source file: {source_path}")
print(f"Rows compared (unique names): {total:,}")
print("\nConfusion Matrix:")
print(f"  TP: {tp:,} | FP: {fp:,}")
print(f"  FN: {fn:,} | TN: {tn:,}")
print("\nMetrics:")
print(f"  Accuracy   : {accuracy:.2f}%")
print(f"  Precision  : {precision:.2f}%")
print(f"  Recall     : {recall:.2f}%")
print(f"  Specificity: {specificity:.2f}%")
print(f"  F1 Score   : {f1:.2f}%")

comparison['error_type'] = np.select(
    [
        (comparison['gold_purdue'] == True) & (comparison['pred_purdue'] == False),
        (comparison['gold_purdue'] == False) & (comparison['pred_purdue'] == True),
    ],
    ['FN', 'FP'],
    default='OK'
)

mismatches = comparison[comparison['error_type'] != 'OK'].copy()
print(f"\nMismatches: {len(mismatches):,}")

if not mismatches.empty:
    show_cols = ['name', 'Purdue', 'gold_purdue', 'pred_purdue', 'error_type']
    if 'verdict' in final_results.columns or 'summary' in final_results.columns:
        enrich = results_df[['name_key'] + [c for c in ['verdict', 'summary', 'pre_llm_stage', 'pre_llm_reason_codes'] if c in results_df.columns]].drop_duplicates('name_key')
        mismatches = mismatches.merge(enrich, on='name_key', how='left')
        show_cols += [c for c in ['verdict', 'pre_llm_stage', 'pre_llm_reason_codes', 'summary'] if c in mismatches.columns]
    display(mismatches[show_cols].sort_values(['error_type', 'name']).reset_index(drop=True))
else:
    print("No mismatches found.")

GOLDEN RECORD COMPARISON (Purdue column)
Source file: E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx
Rows compared (unique names): 389

Confusion Matrix:
  TP: 276 | FP: 4
  FN: 10 | TN: 99

Metrics:
  Accuracy   : 96.40%
  Precision  : 98.57%
  Recall     : 96.50%
  Specificity: 96.12%
  F1 Score   : 97.53%

Mismatches: 14


,name,Purdue,gold_purdue,pred_purdue,error_type,verdict,pre_llm_stage,pre_llm_reason_codes,summary
0,Clark S. Larsen,True,True,False,FN,not_connected,rescue_candidate,edu_signal|strong_relevance|non_affiliation_shape,No evidence found linking Clark S. Larsen to P...
1,Clark Spencer Larsen,True,True,False,FN,not_connected,rescue_candidate,edu_signal|strong_relevance|non_affiliation_shape,No evidence found linking Clark S. Larsen to P...
2,John B. Fenn,True,True,False,FN,not_connected,rescue_candidate,explicit_connection|institution_domain_hit|edu...,"No direct, verifiable link between John B. Fen..."
3,John Bennett Fenn,True,True,False,FN,not_connected,rescue_candidate,explicit_connection|institution_domain_hit|edu...,"No direct, verifiable link between John B. Fen..."
4,Jon Michael Dunn,True,True,False,FN,not_connected,rescue_candidate,institution_domain_hit|edu_signal|person_insti...,"No evidence of a direct alumni, student, facul..."
5,Lawrence D. Brown,True,True,False,FN,not_connected,pass_after_rescue,explicit_connection|authoritative_institution_...,"No valid alumni, faculty, staff, or visiting r..."
6,Miles D. White,True,True,False,FN,not_connected,rescue_candidate,strong_relevance|non_affiliation_shape|low_aut...,No evidence links Miles D. White to Purdue Uni...
7,Ronald A. DeVore,True,True,False,FN,not_connected,pass_strong,authoritative_institution_page|institution_dom...,No direct institutional role or enrollment for...
8,Ronald Alvin DeVore,True,True,False,FN,not_connected,pass_strong,authoritative_institution_page|institution_dom...,No direct institutional role or enrollment for...
9,Thomas S. Huang,True,True,False,FN,not_connected,,,Processing error: Unknown error
